# 02 Baseline and Experiments

В этом ноутбуке сравниваются baseline, несколько моделей, ансамбль и подбор гиперпараметров.

Основной запуск CP2 всё равно лежит в `src/train_cp2.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT / "src"))

from evaluate import evaluate_at_threshold
from preprocessing import build_preprocessor, load_data, make_split, prepare_features

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"

raw_df = load_data(DATA_PATH)
df = prepare_features(raw_df)
X_train, X_val, X_test, y_train, y_val, y_test = make_split(df, drop_duration=True)

X_train.shape, X_val.shape, X_test.shape

: 

## 1. Baseline

Baseline нужен, чтобы сравнить ML-модели с простым решением.  
Здесь используем `DummyClassifier`, который всегда предсказывает самый частый класс.

In [2]:
dummy = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train)),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

dummy.fit(X_train, y_train)
evaluate_at_threshold(dummy, X_val, y_val, "DummyClassifier")

{'model': 'DummyClassifier',
 'threshold': 0.5,
 'f1': 0.0,
 'precision': 0.0,
 'recall': 0.0,
 'roc_auc': 0.5,
 'pr_auc': 0.11700951117009512}

**Вывод:** baseline показывает нижнюю границу качества. Если ML-модель не превосходит baseline по PR-AUC и F1, её нельзя считать полезной.

## 2. Простые первые модели

Для примера обучим LogisticRegression и RandomForest. Полный набор моделей и GridSearchCV запускается через `src/train_cp2.py`.

In [ ]:
logreg = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train)),
        ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42)),
    ]
)

rf = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train)),
        ("classifier", RandomForestClassifier(
            n_estimators=250,
            max_depth=14,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

logreg.fit(X_train, y_train)
rf.fit(X_train, y_train)

rows = [
    evaluate_at_threshold(logreg, X_val, y_val, "LogisticRegression"),
    evaluate_at_threshold(rf, X_val, y_val, "RandomForest"),
]

pd.DataFrame(rows)

: 

**Вывод:** модель выбираем не по одному числу, а по таблице метрик.  
Основной приоритет — PR-AUC, потому что задача несбалансированная.  
F1 используем для выбора threshold, чтобы получить баланс precision и recall.

## 3. Полный запуск CP2

Полный эксперимент с 5 моделями, ансамблем и GridSearchCV запускается командой:

```bash
python src/train_cp2.py
```

## 4. Итоговая таблица CP2-экспериментов

Полный эксперимент запускается через `src/train_cp2.py`. Ниже загружаем сохранённые результаты, чтобы в ноутбуке были видны все модели, GridSearchCV и ансамбль.


In [ ]:
cp2_results = pd.read_csv(PROJECT_ROOT / 'report' / 'cp2_experiment_results.csv')
cp2_results


**Вывод:** в CP2 сравниваются LogisticRegression, RandomForest, ExtraTrees, GradientBoosting, HistGradientBoosting и VotingEnsemble. Основная модель выбирается по PR-AUC, потому что классы несбалансированы и важно хорошо ранжировать клиентов с положительным откликом.


## 5. Итог на test split


In [ ]:
test_results = pd.read_csv(PROJECT_ROOT / 'report' / 'cp2_test_results.csv')
test_results


**Вывод:** финальная модель проверяется на отдельном test split. Threshold подбирается на validation split, а test используется только для финальной оценки. Так мы не подгоняем модель под тестовые данные.
